In [1]:
import pandas as pd, matplotlib, Bio, rapidfuzz, sklearn
print("pandas", pd.__version__, "| all imports OK")

pandas 3.0.3 | all imports OK


## Day 3: load data

In [2]:
import pandas as pd
parts = []
for src in ["A", "B", "C"]:
    d = pd.read_csv(f"../data/raw/lens_{src}.csv")
    d["source"] = src
    parts.append(d)
raw = pd.concat(parts, ignore_index=True)

# one row per document, recording every search that found it
sources = raw.groupby("Lens ID")["source"].agg(lambda s: "".join(sorted(set(s))))
raw = raw.drop_duplicates("Lens ID").copy()
raw["source"] = raw["Lens ID"].map(sources)
print(raw.shape)
print(raw["source"].value_counts())  # e.g. A, B, AB, ABC...

(5655, 38)
source
B      2411
A      2223
AB      823
C        99
BC       37
AC       36
ABC      26
Name: count, dtype: int64


## Sampling data

In [3]:
def show_sample(src, n=10, seed=1):
    s = raw[raw["source"] == src].sample(n, random_state=seed)
    for i, (_, r) in enumerate(s.iterrows(), 1):
        print(f"--- {src} #{i} | {r['Lens ID']}")
        print("TITLE:", r["Title"])
        print("ABSTRACT:", str(r["Abstract"])[:600], "\n")

show_sample("B")

--- B #1 | 191-072-258-411-173
TITLE: DEGRADERS OF CYCLIN-DEPENDENT KINASE 12 (CDK12) AND USES THEREOF
ABSTRACT: Provided herein are bifunctional compounds with a moiety (e.g., lenalidomide, thalidomide) that is a binder of an E3 ubiquitin ligase (e.g., Cereblon) and another moiety that is a binder of a target protein (e.g., kinase (e.g., CDK (e.g., CDK9 and/or CDK12))) to induce degradation of the target protein CDK9 and/or CDK12. Also provided are pharmaceutical compositions comprising the bifunctional compounds, and methods of treating and/or preventing diseases (e.g., proliferative diseases, such as cancers (e.g., ovarian cancer, breast cancer, or prostate cancer))). Provided also are methods of indu 

--- B #2 | 072-430-300-226-557
TITLE: METHOD AND COMPOSITIONS FOR REGULATED ARMORING OF CELLS
ABSTRACT: Provided herein are compositions and methods for regulating expression of effector molecules using regulatable transcription factors and/or activation inducible promoters. In one a

In [4]:
show_sample("C", n=10, seed=2)
show_sample("A", n=20, seed=3)

--- C #1 | 144-597-545-278-042
TITLE: SUPPRESSION OF INFLAMMASOME ACTIVATION
ABSTRACT: The present invention is directed to various methods of suppressing inflammasome activation. For example, the present invention relates to various methods of treating a disease or condition in which inflammation contributes to the disease or condition state. 

--- C #2 | 006-280-716-050-44X
TITLE: GSPT1 DEGRADING AGENT, ANTIBODY CONJUGATE CONTAINING GSPT1 DEGRADING AGENT, AND USE THEREOF
ABSTRACT: Provided are a GSPT1 degrading agent (a domide molecular glue derivative) having an anti-tumor effect and an antibody-drug conjugate thereof, a stereoisomer thereof, a prodrug thereof, a pharmaceutically acceptable salt thereof, a pharmaceutically acceptable solvate thereof, or an isotopic marker thereof, use thereof, and a preparation method therefor. 

--- C #3 | 199-657-291-433-344
TITLE: COMPOUND BASED ON SULFUR/OXYGEN SUBSTITUTED GLUTARIMIDE-BASED ISOINDOLINONE SKELETON AND USE THEREOF
ABSTRACT: The pr

In [5]:
for co in ["Arvinas", "Kymera", "C4 Therapeutics", "Nurix", "Monte Rosa"]:
    n = raw["Applicants"].astype(str).str.contains(co, case=False).sum()
    print(f"{co}: {n} documents")

Arvinas: 491 documents
Kymera: 85 documents
C4 Therapeutics: 135 documents
Nurix: 63 documents
Monte Rosa: 37 documents


## Validation set: 40 hand-checked documents

In [6]:
rows = []
def add(ids, search, relevant):
    rows.extend({"lens_id": i, "search": search, "relevant": relevant} for i in ids)

add(["191-072-258-411-173", "188-163-120-246-705", "175-454-368-633-344", "024-609-250-324-659",
     "138-818-849-452-352", "169-064-459-116-851", "188-441-758-369-664"], "B", True)
add(["072-430-300-226-557", "120-316-870-155-500", "028-085-142-413-970"], "B", False)

add(["023-204-928-463-58X", "129-031-163-399-934", "023-460-981-608-875", "177-039-463-366-074",
     "083-758-315-727-299", "125-760-578-478-428", "112-118-175-452-91X", "121-521-191-818-121",
     "153-420-627-118-360", "129-176-675-883-452", "018-451-649-219-194", "180-223-207-049-932",
     "035-330-858-484-934", "147-580-904-856-624", "096-918-557-758-137"], "A", True)
add(["145-668-337-202-889", "019-298-528-729-604", "104-886-019-309-558",
     "157-889-433-448-277", "076-426-594-781-743"], "A", False)

add(["006-280-716-050-44X", "199-657-291-433-344", "161-235-863-722-880", "019-164-515-782-752",
     "045-428-890-095-806", "136-465-308-013-265", "137-908-426-922-730", "045-513-646-117-15X"], "C", True)
add(["144-597-545-278-042", "001-792-577-993-882"], "C", False)

val = pd.DataFrame(rows)
val.to_csv("../data/validation_search.csv", index=False)
val.groupby(["search", "relevant"]).size()

search  relevant
A       False        5
        True        15
B       False        3
        True         7
C       False        2
        True         8
dtype: int64

## Title + abstract filter (final v4, rules A–H)

In [7]:
## filtering documents to title + abstract only (tight version)
ta = (raw["Title"].fillna("") + " " + raw["Abstract"].fillna("")).str.lower()

A = ta.str.contains(r"protac|proteolysis.targeting chimera|targeted protein degradation|heterobifunctional degrader|bifunctional degrader")
B = ta.str.contains(r"degrad") & ta.str.contains(r"e3 ligase|e3 ubiquitin ligase|ubiquitin ligase")
C = ta.str.contains(r"molecular glue|molecular adhesive") & ta.str.contains(r"degrad")
D = ta.str.contains(r"bifunctional") & ta.str.contains(r"proteolysis|degrad")          # ER-proteolysis style
E = ta.str.contains(r"e3 (?:ubiquitin )?ligase (?:binder|ligand|binding)")             # PROTAC structure wording
F = ta.str.contains(r"(?:selective|protein) degradation agent|degrading agent")
G = ta.str.contains(r"degrader|degrading compound|targeted degradation|proteasom\w* degradation|induc\w* (?:the )?degradation of|heterobifunctional")
H = ta.str.contains(r"targeting chimer|target protein degradation|modulators? of (?:\w+ )?proteolysis|degrading \w+ protein|for (?:the )?degradation of|chemical degradation")
homonym = ta.str.contains(r"coagulat|thrombin|protein [cs]\b")

raw["keep"] = (A | B | C | D | E | F | G | H) & ~homonym
print(raw["keep"].sum(), "of", len(raw), "documents kept")

4533 of 5655 documents kept


In [8]:
## testing against validation set
val = pd.read_csv("../data/validation_search.csv")
val["kept"] = val["lens_id"].map(raw.set_index("Lens ID")["keep"])
print(val.groupby("relevant")["kept"].agg(kept="sum", total="count"))
print("Relevant docs lost:", val[val["relevant"] & ~val["kept"]]["lens_id"].tolist())

          kept  total
relevant             
False        1     10
True        30     30
Relevant docs lost: []


In [9]:
## precision check on whether the kept documents are relevant, 20 unseen documents
fresh = raw[raw["keep"] & ~raw["Lens ID"].isin(val["lens_id"])].sample(20, random_state=31)
for i, (_, r) in enumerate(fresh.iterrows(), 1):
    print(f"--- KEPT #{i} | {r['Lens ID']}\nTITLE: {r['Title']}\nABSTRACT: {str(r['Abstract'])[:400]}\n")

--- KEPT #1 | 116-477-585-831-621
TITLE: SMALL MOLECULE DEGRADERS OF FKBP12 VIA RECRUITMENT OF VON HIPPEL-LINDAU E3 UBIQUITIN LIGASE (VHL) E3 UBIQUITIN LIGASE, AND USES IN dTAG SYSTEMS
ABSTRACT: Disclosed is a dTAG system comprising a small molecule degraders of FKBP12-tagged proteins via recruitment of Von Hippel-Lindau E3 ubiquitin ligase (VHL) E3 ubiquitin ligase and uses thereof.

--- KEPT #2 | 127-542-803-177-531
TITLE: BIFUNCTIONAL DEGRADERS OF GALACTOSE-DEFICIENT IMMUNOGLOBULINS
ABSTRACT: A composition of matter including a deglycosylated IgA-binding moiety, a cellular receptor-binding moiety that binds to hepatocytes or other degrading cells through asialoglycoprotein receptors (ASGPR) on the surface of hepatocytes or other degrading cells in a patient or subject, and optionally, a linker moiety connecting deglycosylated IgA-binding moiety and the cellular receptor-binding moiety,

--- KEPT #3 | 076-360-618-100-430
TITLE: OLIGONUCLEOTIDE-BASED PROTEOLYSIS TARGETING CHIMERA
ABST

**Final filter v4, kept sample (seed 31): 18/20 relevant (90%).** Not relevant: #8 (intein/ADC chemistry), #18 (non-molecule method). The judged documents and verdicts are stored in data/validation_final_v4.csv; the sample shown here is order-proof and may differ.

In [10]:
dropped = raw[~raw["keep"] & ~raw["Lens ID"].isin(val["lens_id"])].sample(20, random_state=32)
for i, (_, r) in enumerate(dropped.iterrows(), 1):
    print(f"--- DROPPED #{i} | {r['Lens ID']}\nTITLE: {r['Title']}\nABSTRACT: {str(r['Abstract'])[:400]}\n")

--- DROPPED #1 | 027-903-856-245-484
TITLE: USE OF A G PROTEIN-COUPLED RECEPTOR KINASE 2 (GRK2) DEGRADATION COMPOUND IN LOWERING BLOOD GLUCOSE, STIMULATING OR INCREASING INSULIN SECRETION, OR PREVENTING OR TREATING A DISEASE OR DISORDER RELATED THERETO
ABSTRACT: Provided is a use of a G protein-coupled receptor kinase 2 (GRK2) degradation compound in lowering blood glucose, stimulating or increasing insulin secretion, or preventing or treating a disease or disorder related thereto. Specifically, provided is a use of a compound represented by Formula (1) that degrades G protein-coupled receptor kinase 2 (GRK2) or a pharmaceutically acceptable salt, an isom

--- DROPPED #2 | 006-339-635-726-640
TITLE: ENGINEERED NK CELLS AND METHODS OF TREATING CANCER
ABSTRACT: The present disclosure relates generally to modified NK cell compositions and methods of using the modified cell compositions in immunotherapy applications. In embodiments, the modified cell express a γ chain and a δ chain. In emb

**Dropped sample (seed 32): 0 clear relevant, 1 borderline (#10, designed polypeptides regulating protein levels).** Earlier tuning rounds (seeds 11/12, 21/22) are recorded in data/SEARCH.md.
verdicts are stored in data/validation_final_v4.csv

In [11]:
kept_y = """191-911-990-355-664 187-628-541-023-797 072-886-707-866-674 151-475-312-524-132
070-646-630-507-888 117-047-840-191-77X 095-446-521-280-152 150-014-586-682-295
197-085-468-441-857 089-638-282-799-65X 147-907-848-969-190 184-102-362-590-088
179-569-890-058-923 018-024-865-528-34X 076-896-148-419-142 167-988-088-348-107
000-479-200-421-705 076-695-313-891-467""".split()
kept_n = ["154-767-241-890-330", "113-702-293-295-571"]
drop_n = """042-254-220-977-141 113-667-896-868-905 012-364-427-359-753 030-136-436-670-453
094-389-868-299-579 154-926-704-623-93X 109-799-146-306-268 105-795-589-264-85X
158-662-165-977-550 038-859-081-337-87X 123-611-774-028-668 156-082-995-218-197
073-084-717-863-356 173-052-461-958-720 150-699-071-675-576 165-665-198-154-670
074-367-330-329-681 112-127-248-521-416 025-306-562-216-92X""".split()
drop_b = ["126-101-011-371-440"]

final = pd.DataFrame(
    [{"lens_id": i, "sample": "kept", "verdict": "Y"} for i in kept_y] +
    [{"lens_id": i, "sample": "kept", "verdict": "N"} for i in kept_n] +
    [{"lens_id": i, "sample": "dropped", "verdict": "N"} for i in drop_n] +
    [{"lens_id": i, "sample": "dropped", "verdict": "borderline"} for i in drop_b])
final.to_csv("../data/validation_final_v4.csv", index=False)
final.groupby(["sample", "verdict"]).size()

sample   verdict   
dropped  N             19
         borderline     1
kept     N              2
         Y             18
dtype: int64

In [12]:
final = pd.read_csv("../data/validation_final_v4.csv")
status = raw.set_index("Lens ID")["keep"]
final["now_kept"] = final["lens_id"].map(status)
print("Missing from data:", final["now_kept"].isna().sum())
print("Kept sample still kept:", final.loc[final["sample"] == "kept", "now_kept"].all())
print("Dropped sample still dropped:", (~final.loc[final["sample"] == "dropped", "now_kept"].astype(bool)).all())

Missing from data: 0
Kept sample still kept: True
Dropped sample still dropped: True


In [13]:
kept = raw[raw["keep"]].copy()
print(kept.shape)  # (4533, ...)

(4533, 39)


In [14]:
for co in ["Arvinas", "Kymera", "C4 Therapeutics", "Nurix", "Monte Rosa"]:
    print(f"{co}: {kept['Applicants'].astype(str).str.contains(co, case=False).sum()} documents")

Arvinas: 490 documents
Kymera: 85 documents
C4 Therapeutics: 135 documents
Nurix: 63 documents
Monte Rosa: 30 documents


In [15]:
print(kept.columns.tolist())
COLS = {
    "id": "Lens ID",
    "title": "Title",
    "abstract": "Abstract",
    "applicants": "Applicants",
    "priority_date": "Earliest Priority Date",
    "priority_numbers": "Priority Numbers",
    "family_members": "Simple Family Members",
    "cited_by": "Cited by Patent Count",
    "cpc": "CPC Classifications",
}
missing = [v for v in COLS.values() if v not in kept.columns]
print("Missing:", missing)

['#', 'Jurisdiction', 'Kind', 'Display Key', 'Lens ID', 'Publication Date', 'Publication Year', 'Application Number', 'Application Date', 'Priority Numbers', 'Earliest Priority Date', 'Title', 'Abstract', 'Applicants', 'Inventors', 'Owners', 'URL', 'Document Type', 'Has Full Text', 'Cites Patent Count', 'Cited by Patent Count', 'Simple Family Size', 'Simple Family Members', 'Simple Family Member Jurisdictions', 'Extended Family Size', 'Extended Family Members', 'Extended Family Member Jurisdictions', 'Sequence Count', 'CPC Classifications', 'IPCR Classifications', 'US Classifications', 'NPL Citation Count', 'NPL Resolved Citation Count', 'NPL Resolved Lens ID(s)', 'NPL Resolved External ID(s)', 'NPL Citations', 'Legal Status', 'source', 'keep']
Missing: []


In [16]:
kept["CPC Classifications"].dropna().str.split(";;").explode().str.strip().str[:4].value_counts().head(10)

CPC Classifications
C07D    29527
A61K    25095
A61P    16712
C07K     5913
C12N     1974
G01N      945
C07C      743
C12Y      443
C07J      400
C07F      382
Name: count, dtype: int64